In [51]:
import pandas as pd
import numpy as np
df = pd.read_csv("retail_data.csv")

In [5]:
print(df.head())

   order_id  order_date order_time       city     product     category  \
0       NaN  2024-05-03   10:45:00  Hyderabad       Watch          NaN   
1       NaN  2024-10-26   17:36:00      Delhi      Mobile  Electronics   
2       NaN  2024-09-02      17:19     Mumbai       Shoes      Fashion   
3    1003.0  08-09-2023   12:16:00     Mumbai  Headphones  Electronics   
4       NaN  23-09-2023   20:54:00  Bangalore      Mobile  Electronics   

   quantity    price payment_method sales_channel  
0       NaN   1000.0           Cash        Online  
1       1.0    600.0           Cash       Offline  
2       1.0  50000.0            NaN       Offline  
3       3.0    500.0            UPI        Online  
4       1.0    500.0           Cash       Offline  


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5300 entries, 0 to 5299
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        3552 non-null   float64
 1   order_date      5300 non-null   object 
 2   order_time      5300 non-null   object 
 3   city            4513 non-null   object 
 4   product         4984 non-null   object 
 5   category        3987 non-null   object 
 6   quantity        3970 non-null   float64
 7   price           2664 non-null   float64
 8   payment_method  3948 non-null   object 
 9   sales_channel   5300 non-null   object 
dtypes: float64(3), object(7)
memory usage: 414.2+ KB


In [11]:
# Finding missing values
print(df.isnull().sum())

order_id          1748
order_date           0
order_time           0
city               787
product            316
category          1313
quantity          1330
price             2636
payment_method    1352
sales_channel        0
dtype: int64


In [5]:
# handling missing values:order_id
mask = df['order_id'].isna()
df.loc[mask,'order_id']= ['ORD_'+ str(i) for i in range(mask.sum())]

C:\Users\jesna\AppData\Local\Temp\ipykernel_14216\3363242229.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['ORD_0', 'ORD_1', 'ORD_2', 'ORD_3', 'ORD_4', 'ORD_5', 'ORD_6', 'ORD_7', 'ORD_8', 'ORD_9', 'ORD_10', 'ORD_11', 'ORD_12', 'ORD_13', 'ORD_14', 'ORD_15', 'ORD_16', 'ORD_17', 'ORD_18', 'ORD_19', 'ORD_20', 'ORD_21', 'ORD_22', 'ORD_23', 'ORD_24', 'ORD_25', 'ORD_26', 'ORD_27', 'ORD_28', 'ORD_29', 'ORD_30', 'ORD_31', 'ORD_32', 'ORD_33', 'ORD_34', 'ORD_35', 'ORD_36', 'ORD_37', 'ORD_38', 'ORD_39', 'ORD_40', 'ORD_41', 'ORD_42', 'ORD_43', 'ORD_44', 'ORD_45', 'ORD_46', 'ORD_47', 'ORD_48', 'ORD_49', 'ORD_50', 'ORD_51', 'ORD_52', 'ORD_53', 'ORD_54', 'ORD_55', 'ORD_56', 'ORD_57', 'ORD_58', 'ORD_59', 'ORD_60', 'ORD_61', 'ORD_62', 'ORD_63', 'ORD_64', 'ORD_65', 'ORD_66', 'ORD_67', 'ORD_68', 'ORD_69', 'ORD_70', 'ORD_71', 'ORD_72', 'ORD_73', 'ORD_74', 'ORD_75', 'ORD_76', 'ORD_77', 'ORD_78', 'ORD_79', 'ORD_80', 'OR

In [7]:
# handling missing values : category
# Step 1: Create product → category mapping
prod_to_cat = (
    df.dropna(subset=['product', 'category'])
      .groupby('product')['category']
      .agg(lambda x: x.mode()[0])
)
# Step 2: Fill missing category using product
df['category'] = df['category'].fillna(df['product'].map(prod_to_cat))
# Step 3: Fill the remaining category Unknown
df['category'] = df['category'].fillna("Unknown")

In [69]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5098 entries, 0 to 5297
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        5098 non-null   object 
 1   order_date      5098 non-null   object 
 2   order_time      5098 non-null   object 
 3   city            5098 non-null   object 
 4   product         5098 non-null   object 
 5   category        5098 non-null   object 
 6   quantity        5098 non-null   float64
 7   price           5098 non-null   float64
 8   payment_method  5098 non-null   object 
 9   sales_channel   5098 non-null   object 
dtypes: float64(2), object(8)
memory usage: 438.1+ KB


In [9]:
# handling missing values: city, product, quantity, payment method
df['city']=df['city'].fillna("Unknown")
df['product']=df['product'].fillna("Unknown Product")
df['quantity']=df['quantity'].fillna(df['quantity'].median())
df['payment_method']=df['payment_method'].fillna("Unknown")

In [11]:
# handling missing price
# Fill using product-level median
df['price'] = df.groupby('product')['price'].transform(
    lambda x: x.fillna(x.median())
)

# Fill remaining with overall median
df['price'] = df['price'].fillna(df['price'].median())


In [13]:
df.isna().sum()

order_id          0
order_date        0
order_time        0
city              0
product           0
category          0
quantity          0
price             0
payment_method    0
sales_channel     0
dtype: int64

In [15]:
#Removing duplicates
df.duplicated().sum()

202

In [65]:
df[df.duplicated()]

,order_id,order_date,order_time,city,product,category,quantity,price,payment_method,sales_channel
5000,3764.0,2023-08-01,12:31:00,Unknown,Mobile,Electronics,3.0,2400.0,Unknown,Offline
5002,4814.0,2024-04-01,15:47:00,Hyderabad,Bag,Accessories,2.0,2400.0,Unknown,Offline
5003,4499.0,04-10-2023,16:59,Mumbai,Unknown Product,Unknown,2.0,2400.0,UPI,Offline
5004,3735.0,2023-07-05,10:42:00,Delhi,Watch,Accessories,2.0,50000.0,UPI,Offline
5006,3701.0,2024-07-15,21:34:00,Unknown,Laptop,Electronics,1.0,2400.0,Unknown,Offline
...,...,...,...,...,...,...,...,...,...,...
5293,3809.0,2023-11-19,10:42:00,Chennai,Watch,Accessories,3.0,2400.0,Unknown,Offline
5294,2399.0,13-03-2023,09:18:00,Pune,Headphones,Electronics,2.0,20000.0,Unknown,Offline
5296,3754.0,2024-08-02,23:27,Delhi,Shoes,Fashion,2.0,5000.0,Card,Offline
5298,3670.0,2023-01-20,08:36:00,Delhi,Shoes,Fashion,2.0,50000.0,Unknown,Online


In [53]:
df=df.drop_duplicates()

In [55]:
df['order_date'].head(10)

0    2024-05-03
1    2024-10-26
2    2024-09-02
3    08-09-2023
4    23-09-2023
5    2023-10-06
6    2024-11-19
7    2024-10-28
8    2024-09-08
9    2024-07-18
Name: order_date, dtype: object

In [57]:
df['order_date_raw'] = df['order_date']
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
mask = df['order_date'].isna()
df.loc[mask,'order_date'] = pd.to_datetime(
    df.loc[mask, 'order_date_raw'],
    errors='coerce',
    dayfirst=True
)

In [59]:
df['order_date'].head(10)

0   2024-05-03
1   2024-10-26
2   2024-09-02
3   2023-09-08
4   2023-09-23
5   2023-10-06
6   2024-11-19
7   2024-10-28
8   2024-09-08
9   2024-07-18
Name: order_date, dtype: datetime64[ns]

In [61]:
df['order_date'].isna().sum()

0

In [69]:
df['order_time_raw'].head()

0    10:45:00
1    17:36:00
2       17:19
3    12:16:00
4    20:54:00
Name: order_time_raw, dtype: object

In [63]:
df['order_time_raw'] = df['order_time']
df['order_time']=pd.to_datetime(df['order_time'],
                               errors='coerce')

C:\Users\jesna\AppData\Local\Temp\ipykernel_14216\2351439657.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['order_time']=pd.to_datetime(df['order_time'],


In [71]:
df['order_time'].head(10)

0   2026-04-23 10:45:00
1   2026-04-23 17:36:00
2   2026-04-23 17:19:00
3   2026-04-23 12:16:00
4   2026-04-23 20:54:00
5   2026-04-23 05:15:00
6   2026-04-23 06:59:00
7   2026-04-23 16:02:00
8   2026-04-23 00:22:00
9   2026-04-23 12:11:00
Name: order_time, dtype: datetime64[ns]

In [73]:
df['order_time'] = df['order_time'].dt.time

In [75]:
df['order_time'].head(10)

0    10:45:00
1    17:36:00
2    17:19:00
3    12:16:00
4    20:54:00
5    05:15:00
6    06:59:00
7    16:02:00
8    00:22:00
9    12:11:00
Name: order_time, dtype: object